# Lamplighter — training a conditional GAN

A plain GAN learns to paint digits, but you can't ask it for a *particular* one.
A **conditional GAN** feeds the class label into both models, so the generator
learns `label → image` and you can request "draw me a 7". This notebook builds
that — two models, each taking the label as a second input — and trains them in
tandem, all from the browser. Training runs *in this kernel*, executing exactly
the `train()` the Training tab shows.

The loop:

1. **Start** a session here → 2. build a **Generator** and a **Discriminator** in
   the **Models** view, each with a **label** input → 3. register MNIST images
   *and* labels with `sess.data(X=X, y=y)` → 4. pick the **Conditional GAN**
   recipe in the **Training** tab (it wires the label into both models for you) →
   5. press **▶ Run**, then pull `sess.models["generator"]` back here and sample a
   digit of your choosing.

> This notebook lives in `examples/`; the first cell puts the repo root on
> `sys.path` so `import lamplighter` resolves.

## 1. Start a session

The first run builds the frontend, then serves the app and opens it in your
browser. Subsequent runs are instant.

In [ ]:
import sys
from pathlib import Path

# The repo root (this notebook runs with examples/ as its cwd).
sys.path.insert(0, str(Path.cwd().parent.resolve()))

import lamplighter

sess = lamplighter.start()
sess.url

## 2. Build the two conditional models

Open the **Models** tab. It starts with one model; use it as the **Generator**,
and add a second for the **Discriminator** with the **＋** in the sidebar.
Double-click a model's name to rename it; the **›** button (or double-clicking its
node) opens its canvas.

The twist versus a plain GAN: **each model takes the label as a second input**,
turned into a vector by an **Embedding** and joined to the main input with a
**Concat**. Add a second **Input** node for the label on each model, and in its
Inspector set **Name** to `label` and **Dtype** to `long` (an Embedding indexes
with integers). Name the main inputs `noise` / `image` too, and keep each model's
`noise`/`image` input *above* its `label` input.

**Generator** (`noise` + `label` → image):

> - Input **Name** `noise`, **Shape** `1, 100`
> - Input **Name** `label`, **Shape** `1`, **Dtype** `long`
>   → **Embedding** (Num Embeddings `10`, Embedding Dim `50`)
> - **Concat** the `noise` input and the Embedding output (**Dim** `1`)
>   → Linear `256` → LeakyReLU → Linear `512` → LeakyReLU → Linear `784` → Tanh →
>   Output

**Discriminator** (image + `label` → one real/fake logit):

> - Input **Name** `image`, **Shape** `1, 784`
> - Input **Name** `label`, **Shape** `1`, **Dtype** `long`
>   → **Embedding** (Num Embeddings `10`, Embedding Dim `50`)
> - **Concat** the `image` input and the Embedding output (**Dim** `1`)
>   → Linear `512` → LeakyReLU → Linear `256` → LeakyReLU → Linear `1` → Output

Set every **LeakyReLU**'s **Negative Slope** to `0.2` (the GAN standard). You
don't set the Linear *input* sizes — Lamplighter infers them from the Concat, so
the first Linear picks up `100 + 50 = 150` (generator) / `784 + 50 = 834`
(discriminator) automatically. Leave the linking to step 4; the recipe draws it.

## 3. Register MNIST — images *and* labels

A conditional GAN conditions on the class, so register both the images and their
labels. Normalize the images to `[-1, 1]` to match the generator's `Tanh` output;
keep the labels as the integer class indices the Embedding expects.

In [ ]:
import torch
from torchvision import datasets

mnist = datasets.MNIST(root="./data", train=True, download=True)
# Flatten 28x28 -> 784 and scale [0, 255] -> [-1, 1] (matches the Tanh output).
X = (mnist.data.float() / 127.5 - 1.0).view(-1, 784)
y = mnist.targets  # integer class labels 0..9 — the conditioning signal
torch.manual_seed(0)
idx = torch.randperm(len(X))[:8000]  # subsample for a snappy CPU demo
X, y = X[idx], y[idx]

sess.data(X=X, y=y)  # images AND labels — the label conditions both models

## 4. Assign the conditional-GAN roles

Switch to the **Training** tab and choose the **Conditional GAN** recipe. Assign
the **Generator** and **Discriminator** roles to your two models (each gets its
own **Learning Rate** — `0.0002` is the classic default).

Assigning the roles wires everything on the **System** canvas for you:

- a **Noise** node into the generator's `noise` port, and
- a **Data** node whose output now splits into two pins — **`x`** (the images)
  feeding the discriminator's `image` port, and **`y`** (the label) feeding the
  `label` port of *both* models.

That `y` wire into both models is the whole point: the same class label
conditions the image the generator paints and the judgement the discriminator
makes. Trace the wires and you're looking at exactly the `generator(noise,
labels)` / `discriminator(image, labels)` calls the recipe generates.

## 5. Pick the data and train

On the **System** canvas, select the **Data** node, keep the source on **memory**,
and pick `X` under **Input(s)** and `y` under **Target(s)** (hit **↻ refresh**
first if they're not listed). There's no validation split for an adversarial loop.

Then return to **Training** and set **Epochs** — a conditional GAN, like a plain
one, needs room: **~250** (about **90 seconds** on CPU for this 8k subset) is
where legible, *controllable* digits emerge. Press **▶ Run**.

Two curves stream in below the code: `d_loss` and `g_loss`. Watch for them to
*settle* into an equilibrium (roughly `1.0` / `1.4`) rather than one running away
— that balance is when the samples start to look like the digits you ask for.

In [ ]:
# The run's per-epoch losses, and the two trained models (keyed by role).
print("epochs trained:", len(sess.history["g_loss"]))
generator = sess.models["generator"]
discriminator = sess.models["discriminator"]
generator

## 6. Sample a digit on demand

Here's the payoff a plain GAN can't give you: pass the generator both noise **and
a label**, and it paints that class. The grid below asks for every digit 0–9 (one
row each) — read down the rows to see the model obeying the label.

In [ ]:
import matplotlib.pyplot as plt
import torch.nn as nn

generator = generator.to("cpu").eval()
# Read the latent size and class count straight off the model: the first Linear
# sees noise+embedding, so the noise width is its in_features minus the embedding.
emb = next(m for m in generator.modules() if isinstance(m, nn.Embedding))
first_linear = next(m for m in generator.modules() if isinstance(m, nn.Linear))
latent = first_linear.in_features - emb.embedding_dim
num_classes = emb.num_embeddings

torch.manual_seed(0)
per_class = 8
labels = torch.arange(num_classes).repeat_interleave(per_class)  # 0,0,..,1,1,..
noise = torch.randn(len(labels), latent)
with torch.no_grad():
    fakes = generator(noise, labels).reshape(-1, 28, 28)  # each pixel in [-1, 1]

fig, axes = plt.subplots(num_classes, per_class, figsize=(per_class, num_classes))
for i, (ax, img) in enumerate(zip(axes.ravel(), fakes)):
    ax.imshow(img, cmap="gray", vmin=-1, vmax=1)
    ax.set_xticks([]); ax.set_yticks([])
    if i % per_class == 0:
        ax.set_ylabel(str(i // per_class), rotation=0, ha="right", va="center", fontsize=12)
fig.suptitle("conditional samples — each row is a requested digit")
plt.tight_layout()
plt.show()

## 7. Keep both models

The two models save together. `sess.checkpoint("mnist-cgan")` keeps the run in the
app's **Checkpoints** strip (and you can **▶ Resume** it toward more epochs);
**⬇ Weights** — or `sess.save_checkpoint(path)` — writes one self-contained `.pt`
holding *both* models. Reload a single model from it by role:

In [ ]:
sess.save_checkpoint("mnist-cgan.pt")

# A multi-model checkpoint holds every model; pick one by its role name.
generator2, snapshot = lamplighter.load_checkpoint("mnist-cgan.pt", model="generator")
with torch.no_grad():
    identical = torch.equal(generator2(noise, labels), generator(noise, labels))
print("reloaded generator matches:", identical)  # same weights, same output

## 8. Tear down

Stops the server thread. (A kernel restart also stops it.)

In [ ]:
lamplighter.stop()